In [1]:
import pandas as pd

bat = pd.read_csv('Batting_Final.csv')
bowl = pd.read_csv('Bowling_Final.csv')
mm = pd.read_csv('Match_Master.csv')

bat.shape, bowl.shape, mm.shape

((676, 15), (559, 17), (31, 11))

In [2]:
for col in ['Team', 'Batter']:
    bat[col] = bat[col].astype(str).str.strip()
for col in ['Team', 'Bowler']:
    bowl[col] = bowl[col].astype(str).str.strip()

bat = bat[~bat['Batter'].str.contains(r'^-+$', regex=True, na=False)]
bowl = bowl[~bowl['Bowler'].str.contains(r'^-+$', regex=True, na=False)]
bat = bat.drop_duplicates().reset_index(drop=True)
bowl = bowl.drop_duplicates().reset_index(drop=True)

bat.shape, bowl.shape

((676, 15), (528, 17))

In [3]:
mm_small = mm[['Match_ID', 'Tournament', 'Ground', 'Date', 'Result', 'Team 1', 'Team 2']]
bat = bat.merge(mm_small, on='Match_ID', how='left')
bowl = bowl.merge(mm_small, on='Match_ID', how='left')

def opposition(row):
    if row['Team'] == row['Team 1']:
        return row['Team 2']
    return row['Team 1']

bat['Opposition'] = bat.apply(opposition, axis=1)
bowl['Opposition'] = bowl.apply(opposition, axis=1)

bat[['Match_ID', 'Batter', 'Team', 'Opposition', 'Tournament']].head()

,Match_ID,Batter,Team,Opposition,Tournament
0,1,Shardul,Parsi Gymkhana U14,Vijay Cricket Club U14,30 Yca Series Match
1,1,Nirav S,Parsi Gymkhana U14,Vijay Cricket Club U14,30 Yca Series Match
2,1,Sanmit Shetane,Parsi Gymkhana U14,Vijay Cricket Club U14,30 Yca Series Match
3,1,Avaneesh Suradkar,Parsi Gymkhana U14,Vijay Cricket Club U14,30 Yca Series Match
4,1,Shreyan Ranaware,Parsi Gymkhana U14,Vijay Cricket Club U14,30 Yca Series Match


In [4]:
def build_batting_career(bat):
    rows = []
    for player, grp in bat.groupby('Batter'):
        dismissed = grp[~grp['Dismissal_Type'].isin(['Not Out', 'Retired Hurt'])]
        not_outs = len(grp) - len(dismissed)
        runs = grp['Runs'].sum()
        balls = grp['Balls'].sum()
        innings = len(grp)
        outs = len(dismissed)
        avg = round(runs / outs, 2) if outs > 0 else runs
        sr = round(runs / balls * 100, 2) if balls > 0 else 0
        rows.append({
            'Player': player,
            'Team': grp['Team'].iloc[-1],
            'Matches': grp['Match_ID'].nunique(),
            'Innings': innings,
            'Runs': runs,
            'Balls': balls,
            'Highest Score': grp['Runs'].max(),
            'Not Outs': not_outs,
            'Average': avg,
            'Strike Rate': sr,
            '4s': grp['4s'].sum(),
            '6s': grp['6s'].sum(),
        })
    return pd.DataFrame(rows)

bat_career = build_batting_career(bat)
bat_career.sort_values('Runs', ascending=False).head(10)

,Player,Team,Matches,Innings,Runs,Balls,Highest Score,Not Outs,Average,Strike Rate,4s,6s
59,Atharv Jadhavar,Mavericks Vijay U14,28,30,973,1393,144,5,38.92,69.85,137,2
150,Ranvijay A Patil,Mavericks Vijay U14,18,18,432,577,69,0,24.00,74.87,60,0
32,Anargh Jain,Vijay,20,20,430,551,63,0,21.50,78.04,74,2
25,Advit Suhas Wagh,Mavericks Vijay U14,20,20,428,589,81,0,21.40,72.67,73,6
118,Mitesh Nilesh Shinde,Mavericks Vijay U14,15,15,364,562,84,3,30.33,64.77,58,0
250,Vihan Yevankar,Vijay Cricket Club U14,4,4,295,275,174,0,73.75,107.27,49,1
196,Shivraj Sarde,Vijay Cricket Club U14,7,7,265,217,101,1,44.17,122.12,44,6
170,Samanyu Pal,Mavericks Vijay U14,8,24,258,498,45,0,10.75,51.81,36,0
114,Maanvendra Singh Sengar,Vijay Cricket Club U14,22,22,227,672,20,1,10.81,33.78,24,0
24,Advik Shrirao,Vijay,7,7,193,284,78,0,27.57,67.96,30,1


In [5]:
def build_bowling_career(bowl):
    rows = []
    for player, grp in bowl.groupby('Bowler'):
        overs = grp['overs'].sum()
        balls = grp['balls'].sum()
        runs = grp['runs'].sum()
        wkts = grp['wickets'].sum()
        econ = round(runs / overs, 2) if overs > 0 else 0
        avg = round(runs / wkts, 2) if wkts > 0 else 0
        sr = round(balls / wkts, 2) if wkts > 0 else 0
        dot_pct = round(grp['0s'].sum() / balls * 100, 2) if balls > 0 else 0
        rows.append({
            'Player': player,
            'Team': grp['Team'].iloc[-1],
            'Matches': grp['Match_ID'].nunique(),
            'Overs': round(overs, 1),
            'Runs Conceded': runs,
            'Wickets': wkts,
            'Economy': econ,
            'Bowling Average': avg,
            'Bowling SR': sr,
            'Dot %': dot_pct,
        })
    return pd.DataFrame(rows)

bowl_career = build_bowling_career(bowl)
bowl_career.sort_values('Wickets', ascending=False).head(10)

,Player,Team,Matches,Overs,Runs Conceded,Wickets,Economy,Bowling Average,Bowling SR,Dot %
52,Arush Dinde,Vijay Cricket Club U14,15,116.3,263.0,45.0,2.26,5.84,10.58,123.95
173,Satyajeet Bhumkar,Vijay,17,126.0,520.0,23.0,4.13,22.61,19.83,120.39
190,Shrey Joshi,Mavericks Vijay U14,12,72.3,317.0,18.0,4.38,17.61,8.33,192.67
4,Aahan Thanekar,Vijay Cricket Club U14,21,119.1,530.0,17.0,4.45,31.18,13.41,216.23
139,Rajveer Mehra,Mavericks,12,71.7,319.0,17.0,4.45,18.76,17.06,99.31
233,Yugveer,Mavericks Cricket Academy,17,108.6,539.0,14.0,4.96,38.50,29.14,112.75
70,Daksh Khobragade,Mavericks Cricket Academy,7,56.2,163.0,11.0,2.90,14.82,22.55,106.45
110,Mitesh Nilesh Shinde,Mavericks Vijay U14,15,61.4,291.0,11.0,4.74,26.45,20.18,116.22
21,Advay Newase,Mavericks Vijay U14,13,79.2,340.0,10.0,4.29,34.00,24.00,136.67
116,Omaansh Bora,Mavericks Vijay U14,12,69.0,252.0,9.0,3.65,28.00,28.00,118.65


In [6]:
player = 'Vicky'
display(bat_career[bat_career.Player == player])
display(bowl_career[bowl_career.Player == player])
display(bat[bat.Batter == player][['Match_ID', 'Team', 'Opposition', 'Runs', 'Balls', 'SR', 'Dismissal_Type']])

,Player,Team,Matches,Innings,Runs,Balls,Highest Score,Not Outs,Average,Strike Rate,4s,6s
242,Vicky,Parsi Gymkhana U14,1,1,72,66,72,1,72.0,109.09,13,0


,Player,Team,Matches,Overs,Runs Conceded,Wickets,Economy,Bowling Average,Bowling SR,Dot %
221,Vicky,Parsi Gymkhana U14,1,13.2,57.0,4.0,4.32,14.25,0.0,0.0


,Match_ID,Team,Opposition,Runs,Balls,SR,Dismissal_Type
7,1,Parsi Gymkhana U14,Vijay Cricket Club U14,72,66,109.09,Retired Hurt


In [7]:
###   original code 


# %%writefile app.py
# import streamlit as st
# import pandas as pd
# import plotly.express as px
# import plotly.graph_objects as go

# st.set_page_config(page_title="Cricket Analytics Dashboard", page_icon="🏏", layout="wide")


# # ---------------------------------------------------------------------------
# # DATA LOADING + CLEANUP
# # ---------------------------------------------------------------------------
# @st.cache_data
# def load_data():
#     bat = pd.read_csv("Batting_Final.csv")
#     bowl = pd.read_csv("Bowling_Final.csv")
#     mm = pd.read_csv("Match_Master.csv")

#     for col in ["Team", "Batter"]:
#         bat[col] = bat[col].astype(str).str.strip()
#     for col in ["Team", "Bowler"]:
#         bowl[col] = bowl[col].astype(str).str.strip()

#     bat = bat[~bat["Batter"].str.contains(r"^-+$", regex=True, na=False)]
#     bowl = bowl[~bowl["Bowler"].str.contains(r"^-+$", regex=True, na=False)]
#     bat = bat.drop_duplicates().reset_index(drop=True)
#     bowl = bowl.drop_duplicates().reset_index(drop=True)

#     mm_small = mm[["Match_ID", "Tournament", "Ground", "Date", "Result", "Team 1", "Team 2"]]
#     bat = bat.merge(mm_small, on="Match_ID", how="left")
#     bowl = bowl.merge(mm_small, on="Match_ID", how="left")

#     def opposition(row):
#         if row["Team"] == row["Team 1"]:
#             return row["Team 2"]
#         return row["Team 1"]

#     bat["Opposition"] = bat.apply(opposition, axis=1)
#     bowl["Opposition"] = bowl.apply(opposition, axis=1)

#     return bat, bowl, mm


# bat, bowl, mm = load_data()


# # ---------------------------------------------------------------------------
# # CAREER AGGREGATES
# # ---------------------------------------------------------------------------
# @st.cache_data
# def build_batting_career(bat):
#     rows = []
#     for player, grp in bat.groupby("Batter"):
#         dismissed = grp[~grp["Dismissal_Type"].isin(["Not Out", "Retired Hurt"])]
#         not_outs = len(grp) - len(dismissed)
#         runs = grp["Runs"].sum()
#         balls = grp["Balls"].sum()
#         innings = len(grp)
#         outs = len(dismissed)
#         avg = round(runs / outs, 2) if outs > 0 else runs
#         sr = round(runs / balls * 100, 2) if balls > 0 else 0
#         rows.append({
#             "Player": player,
#             "Team": grp["Team"].iloc[-1],
#             "Matches": grp["Match_ID"].nunique(),
#             "Innings": innings,
#             "Runs": runs,
#             "Balls": balls,
#             "Highest Score": grp["Runs"].max(),
#             "Not Outs": not_outs,
#             "Average": avg,
#             "Strike Rate": sr,
#             "4s": grp["4s"].sum(),
#             "6s": grp["6s"].sum(),
#         })
#     return pd.DataFrame(rows)


# @st.cache_data
# def build_bowling_career(bowl):
#     rows = []
#     for player, grp in bowl.groupby("Bowler"):
#         overs = grp["overs"].sum()
#         balls = grp["balls"].sum()
#         runs = grp["runs"].sum()
#         wkts = grp["wickets"].sum()
#         econ = round(runs / overs, 2) if overs > 0 else 0
#         avg = round(runs / wkts, 2) if wkts > 0 else 0
#         sr = round(balls / wkts, 2) if wkts > 0 else 0
#         dot_pct = round(grp["0s"].sum() / balls * 100, 2) if balls > 0 else 0
#         rows.append({
#             "Player": player,
#             "Team": grp["Team"].iloc[-1],
#             "Matches": grp["Match_ID"].nunique(),
#             "Overs": round(overs, 1),
#             "Runs Conceded": runs,
#             "Wickets": wkts,
#             "Economy": econ,
#             "Bowling Average": avg,
#             "Bowling SR": sr,
#             "Dot %": dot_pct,
#         })
#     return pd.DataFrame(rows)


# bat_career = build_batting_career(bat)
# bowl_career = build_bowling_career(bowl)

# all_players = sorted(set(bat["Batter"]).union(set(bowl["Bowler"])))
# all_teams = sorted(bat["Team"].dropna().unique())
# all_tournaments = sorted(mm["Tournament"].dropna().unique())


# # ---------------------------------------------------------------------------
# # SIDEBAR NAVIGATION
# # ---------------------------------------------------------------------------
# st.sidebar.title("🏏 Cricket Analytics")
# page = st.sidebar.radio("Go to", ["Home", "Player Profile", "Compare Players", "Team Analysis"])

# search = st.sidebar.text_input("🔍 Search Player")
# if search:
#     matches = [p for p in all_players if search.lower() in p.lower()]
#     if matches:
#         st.sidebar.write(matches[:10])


# # ---------------------------------------------------------------------------
# # PAGE: HOME
# # ---------------------------------------------------------------------------
# def show_home():
#     st.title("🏏 Cricket Analytics Dashboard")

#     c1, c2, c3, c4 = st.columns(4)
#     c1.metric("Total Players", len(all_players))
#     c2.metric("Total Matches", mm["Match_ID"].nunique())
#     c3.metric("Total Runs", int(bat["Runs"].sum()))
#     c4.metric("Total Wickets", int(bowl["wickets"].sum()))

#     st.divider()
#     col1, col2 = st.columns(2)

#     with col1:
#         st.subheader("Top Run Scorers")
#         top_runs = bat_career.sort_values("Runs", ascending=False).head(10)
#         fig = px.bar(top_runs, x="Runs", y="Player", orientation="h", color="Runs")
#         fig.update_layout(yaxis={"categoryorder": "total ascending"})
#         st.plotly_chart(fig, width='stretch')

#     with col2:
#         st.subheader("Top Wicket Takers")
#         top_wkts = bowl_career.sort_values("Wickets", ascending=False).head(10)
#         fig = px.bar(top_wkts, x="Wickets", y="Player", orientation="h", color="Wickets")
#         fig.update_layout(yaxis={"categoryorder": "total ascending"})
#         st.plotly_chart(fig, width='stretch')


# # ---------------------------------------------------------------------------
# # PAGE: PLAYER PROFILE
# # ---------------------------------------------------------------------------
# def show_player_profile():
#     st.title("Player Profile")

#     default_player = search if search in all_players else all_players[0]
#     player = st.sidebar.selectbox("Select Player", all_players, index=all_players.index(default_player))

#     bcareer = bat_career[bat_career["Player"] == player]
#     wcareer = bowl_career[bowl_career["Player"] == player]

#     st.header(player)

#     c1, c2, c3, c4 = st.columns(4)
#     if not bcareer.empty:
#         row = bcareer.iloc[0]
#         c1.metric("Matches (Batting)", row["Matches"])
#         c2.metric("Runs", row["Runs"])
#         c3.metric("Highest Score", row["Highest Score"])
#         c4.metric("Batting Average", row["Average"])
#         c5, c6 = st.columns(2)
#         c5.metric("Strike Rate", row["Strike Rate"])
#         c6.metric("4s / 6s", f"{row['4s']} / {row['6s']}")
#     else:
#         st.info("No batting record for this player.")

#     if not wcareer.empty:
#         st.subheader("Bowling")
#         row = wcareer.iloc[0]
#         b1, b2, b3, b4, b5 = st.columns(5)
#         b1.metric("Overs", row["Overs"])
#         b2.metric("Wickets", int(row["Wickets"]))
#         b3.metric("Economy", row["Economy"])
#         b4.metric("Bowling Average", row["Bowling Average"])
#         b5.metric("Dot %", row["Dot %"])

#     st.divider()

#     st.subheader("Match History (Batting)")
#     player_bat = bat[bat["Batter"] == player][
#         ["Match_ID", "Team", "Opposition", "Runs", "Balls", "4s", "6s", "SR", "Dismissal_Type"]
#     ].sort_values("Match_ID")
#     st.dataframe(player_bat, width='stretch', hide_index=True)

#     match_ids = player_bat["Match_ID"].tolist()
#     if match_ids:
#         chosen_match = st.selectbox("Select a Match to view full scorecard", match_ids)
#         show_scorecard(chosen_match)

#     st.divider()

#     st.subheader("Charts")
#     ch1, ch2 = st.columns(2)

#     with ch1:
#         fig = px.line(player_bat, x="Match_ID", y="Runs", markers=True, title="Runs per Match")
#         st.plotly_chart(fig, width='stretch')

#     with ch2:
#         fig = px.line(player_bat, x="Match_ID", y="SR", markers=True, title="Strike Rate Trend")
#         st.plotly_chart(fig, width='stretch')

#     ch3, ch4 = st.columns(2)
#     with ch3:
#         player_bowl = bowl[bowl["Bowler"] == player][["Match_ID", "wickets"]].sort_values("Match_ID")
#         if not player_bowl.empty:
#             fig = px.bar(player_bowl, x="Match_ID", y="wickets", title="Wickets per Match")
#             st.plotly_chart(fig, width='stretch')
#         else:
#             st.info("No bowling data.")

#     with ch4:
#         dismissal_counts = player_bat["Dismissal_Type"].value_counts().reset_index()
#         dismissal_counts.columns = ["Dismissal_Type", "Count"]
#         if not dismissal_counts.empty:
#             fig = px.pie(dismissal_counts, names="Dismissal_Type", values="Count", title="Dismissal Breakdown")
#             st.plotly_chart(fig, width='stretch')

#     ch5, ch6 = st.columns(2)
#     with ch5:
#         vs_opp = player_bat.groupby("Opposition")["Runs"].sum().reset_index().sort_values("Runs", ascending=False)
#         fig = px.bar(vs_opp, x="Opposition", y="Runs", title="Runs vs Opposition")
#         st.plotly_chart(fig, width='stretch')

#     with ch6:
#         vs_ground = bat[bat["Batter"] == player].groupby("Ground")["Runs"].sum().reset_index().sort_values(
#             "Runs", ascending=False)
#         if not vs_ground.empty:
#             fig = px.bar(vs_ground, x="Ground", y="Runs", title="Runs vs Ground")
#             st.plotly_chart(fig, width='stretch')


# def show_scorecard(match_id):
#     st.markdown(f"### Scorecard — Match {match_id}")
#     info = mm[mm["Match_ID"] == match_id].iloc[0]
#     st.write(f"**{info['Team 1']} vs {info['Team 2']}** | {info['Tournament']} | Result: {info['Result']}")

#     for team in [info["Team 1"], info["Team 2"]]:
#         st.markdown(f"**{team} — Batting**")
#         team_bat = bat[(bat["Match_ID"] == match_id) & (bat["Team"] == team)][
#             ["Batter", "Runs", "Balls", "4s", "6s", "SR", "Dismissal"]
#         ]
#         st.dataframe(team_bat, width='stretch', hide_index=True)

#         st.markdown(f"**{team} — Bowling**")
#         team_bowl = bowl[(bowl["Match_ID"] == match_id) & (bowl["Team"] == team)][
#             ["Bowler", "overs", "maidens", "runs", "wickets", "Economy"]
#         ]
#         st.dataframe(team_bowl, width='stretch', hide_index=True)


# # ---------------------------------------------------------------------------
# # PAGE: COMPARE PLAYERS
# # ---------------------------------------------------------------------------
# def show_compare():
#     st.title("Player Compare")

#     col1, col2 = st.columns(2)
#     player_a = col1.selectbox("Player A", all_players, index=0)
#     player_b = col2.selectbox("Player B", all_players, index=1 if len(all_players) > 1 else 0)

#     a_bat = bat_career[bat_career["Player"] == player_a]
#     b_bat = bat_career[bat_career["Player"] == player_b]
#     a_bowl = bowl_career[bowl_career["Player"] == player_a]
#     b_bowl = bowl_career[bowl_career["Player"] == player_b]

#     st.subheader("Batting Comparison")
#     metrics = ["Runs", "Average", "Strike Rate", "Highest Score"]
#     comp_rows = []
#     for m in metrics:
#         a_val = a_bat[m].iloc[0] if not a_bat.empty else 0
#         b_val = b_bat[m].iloc[0] if not b_bat.empty else 0
#         comp_rows.append({"Metric": m, player_a: a_val, player_b: b_val})
#     comp_df = pd.DataFrame(comp_rows)
#     st.dataframe(comp_df, width='stretch', hide_index=True)

#     fig = go.Figure()
#     fig.add_trace(go.Bar(name=player_a, x=comp_df["Metric"], y=comp_df[player_a]))
#     fig.add_trace(go.Bar(name=player_b, x=comp_df["Metric"], y=comp_df[player_b]))
#     fig.update_layout(barmode="group", title="Batting: Head to Head")
#     st.plotly_chart(fig, width='stretch')

#     st.subheader("Bowling Comparison")
#     b_metrics = ["Wickets", "Economy", "Bowling Average", "Dot %"]
#     comp_rows2 = []
#     for m in b_metrics:
#         a_val = a_bowl[m].iloc[0] if not a_bowl.empty else 0
#         b_val = b_bowl[m].iloc[0] if not b_bowl.empty else 0
#         comp_rows2.append({"Metric": m, player_a: a_val, player_b: b_val})
#     comp_df2 = pd.DataFrame(comp_rows2)
#     st.dataframe(comp_df2, width='stretch', hide_index=True)

#     fig2 = go.Figure()
#     fig2.add_trace(go.Bar(name=player_a, x=comp_df2["Metric"], y=comp_df2[player_a]))
#     fig2.add_trace(go.Bar(name=player_b, x=comp_df2["Metric"], y=comp_df2[player_b]))
#     fig2.update_layout(barmode="group", title="Bowling: Head to Head")
#     st.plotly_chart(fig2, width='stretch')


# # ---------------------------------------------------------------------------
# # PAGE: TEAM ANALYSIS
# # ---------------------------------------------------------------------------
# def show_team_analysis():
#     st.title("Team Analysis")
#     team = st.sidebar.selectbox("Select Team", all_teams)

#     st.header(team)
#     team_bat = bat_career[bat_career["Team"] == team].sort_values("Runs", ascending=False)
#     team_bowl = bowl_career[bowl_career["Team"] == team].sort_values("Wickets", ascending=False)

#     c1, c2 = st.columns(2)
#     with c1:
#         st.subheader("Batting Contribution")
#         fig = px.pie(team_bat, names="Player", values="Runs", title="Runs Share by Player")
#         st.plotly_chart(fig, width='stretch')
#         st.dataframe(team_bat, width='stretch', hide_index=True)

#     with c2:
#         st.subheader("Bowling Contribution")
#         fig = px.pie(team_bowl, names="Player", values="Wickets", title="Wickets Share by Player")
#         st.plotly_chart(fig, width='stretch')
#         st.dataframe(team_bowl, width='stretch', hide_index=True)


# # ---------------------------------------------------------------------------
# # ROUTER
# # ---------------------------------------------------------------------------
# if page == "Home":
#     show_home()
# elif page == "Player Profile":
#     show_player_profile()
# elif page == "Compare Players":
#     show_compare()
# elif page == "Team Analysis":
#     show_team_analysis()


Overwriting app.py


In [7]:
# %%writefile app.py
# import streamlit as st
# import pandas as pd
# import plotly.express as px
# import plotly.graph_objects as go

# st.set_page_config(page_title="Cricket Analytics Dashboard", page_icon="🏏", layout="wide")


# # ---------------------------------------------------------------------------
# # DATA LOADING + CLEANUP
# # ---------------------------------------------------------------------------
# @st.cache_data
# def load_data():
#     bat = pd.read_csv("Batting_Final.csv")
#     bowl = pd.read_csv("Bowling_Final.csv")
#     mm = pd.read_csv("Match_Master.csv")

#     for col in ["Team", "Batter"]:
#         bat[col] = bat[col].astype(str).str.strip()
#     for col in ["Team", "Bowler"]:
#         bowl[col] = bowl[col].astype(str).str.strip()

#     bat = bat[~bat["Batter"].str.contains(r"^-+$", regex=True, na=False)]
#     bowl = bowl[~bowl["Bowler"].str.contains(r"^-+$", regex=True, na=False)]
#     bat = bat.drop_duplicates().reset_index(drop=True)
#     bowl = bowl.drop_duplicates().reset_index(drop=True)

#     mm_small = mm[["Match_ID", "Tournament", "Ground", "Date", "Result", "Team 1", "Team 2"]]
#     bat = bat.merge(mm_small, on="Match_ID", how="left")
#     bowl = bowl.merge(mm_small, on="Match_ID", how="left")

#     def opposition(row):
#         if row["Team"] == row["Team 1"]:
#             return row["Team 2"]
#         return row["Team 1"]

#     bat["Opposition"] = bat.apply(opposition, axis=1)
#     bowl["Opposition"] = bowl.apply(opposition, axis=1)

#     return bat, bowl, mm


# bat, bowl, mm = load_data()


# # ---------------------------------------------------------------------------
# # CAREER AGGREGATES
# # ---------------------------------------------------------------------------
# @st.cache_data
# def build_batting_career(bat):
#     rows = []
#     for player, grp in bat.groupby("Batter"):
#         dismissed = grp[~grp["Dismissal_Type"].isin(["Not Out", "Retired Hurt"])]
#         not_outs = len(grp) - len(dismissed)
#         runs = grp["Runs"].sum()
#         balls = grp["Balls"].sum()
#         innings = len(grp)
#         outs = len(dismissed)
#         avg = round(runs / outs, 2) if outs > 0 else runs
#         sr = round(runs / balls * 100, 2) if balls > 0 else 0
#         rows.append({
#             "Player": player,
#             "Team": grp["Team"].iloc[-1],
#             "Matches": grp["Match_ID"].nunique(),
#             "Innings": innings,
#             "Runs": runs,
#             "Balls": balls,
#             "Highest Score": grp["Runs"].max(),
#             "Not Outs": not_outs,
#             "Average": avg,
#             "Strike Rate": sr,
#             "4s": grp["4s"].sum(),
#             "6s": grp["6s"].sum(),
#         })
#     return pd.DataFrame(rows)


# @st.cache_data
# def build_bowling_career(bowl):
#     rows = []
#     for player, grp in bowl.groupby("Bowler"):
#         overs = grp["overs"].sum()
#         balls = grp["balls"].sum()
#         runs = grp["runs"].sum()
#         wkts = grp["wickets"].sum()
#         econ = round(runs / overs, 2) if overs > 0 else 0
#         avg = round(runs / wkts, 2) if wkts > 0 else 0
#         sr = round(balls / wkts, 2) if wkts > 0 else 0
#         dot_pct = round(grp["0s"].sum() / balls * 100, 2) if balls > 0 else 0
#         rows.append({
#             "Player": player,
#             "Team": grp["Team"].iloc[-1],
#             "Matches": grp["Match_ID"].nunique(),
#             "Overs": round(overs, 1),
#             "Runs Conceded": runs,
#             "Wickets": wkts,
#             "Economy": econ,
#             "Bowling Average": avg,
#             "Bowling SR": sr,
#             "Dot %": dot_pct,
#         })
#     return pd.DataFrame(rows)


# bat_career = build_batting_career(bat)
# bowl_career = build_bowling_career(bowl)

# all_players = sorted(set(bat["Batter"]).union(set(bowl["Bowler"])))
# all_teams = sorted(list(set(bat["Team"].dropna().unique()).union(set(bowl["Team"].dropna().unique()))))
# all_tournaments = sorted(mm["Tournament"].dropna().unique())


# # ---------------------------------------------------------------------------
# # SIDEBAR NAVIGATION
# # ---------------------------------------------------------------------------
# st.sidebar.title("🏏 Cricket Analytics")
# page = st.sidebar.radio("Go to", ["Home", "Player Profile", "Compare Players", "Team Analysis"])

# search = st.sidebar.text_input("🔍 Search Player")
# if search:
#     matches = [p for p in all_players if search.lower() in p.lower()]
#     if matches:
#         st.sidebar.write(matches[:10])


# # ---------------------------------------------------------------------------
# # PAGE: HOME
# # ---------------------------------------------------------------------------
# def show_home():
#     st.title("🏏 Cricket Analytics Dashboard")

#     c1, c2, c3, c4 = st.columns(4)
#     c1.metric("Total Players", len(all_players))
#     c2.metric("Total Matches", mm["Match_ID"].nunique())
#     c3.metric("Total Runs", int(bat["Runs"].sum()))
#     c4.metric("Total Wickets", int(bowl["wickets"].sum()))

#     st.divider()
#     col1, col2 = st.columns(2)

#     with col1:
#         st.subheader("Top Run Scorers")
#         top_runs = bat_career.sort_values("Runs", ascending=False).head(10)
#         fig = px.bar(top_runs, x="Runs", y="Player", orientation="h", color="Runs")
#         fig.update_layout(yaxis={"categoryorder": "total ascending"})
#         st.plotly_chart(fig, width='stretch')

#     with col2:
#         st.subheader("Top Wicket Takers")
#         top_wkts = bowl_career.sort_values("Wickets", ascending=False).head(10)
#         fig = px.bar(top_wkts, x="Wickets", y="Player", orientation="h", color="Wickets")
#         fig.update_layout(yaxis={"categoryorder": "total ascending"})
#         st.plotly_chart(fig, width='stretch')


# # ---------------------------------------------------------------------------
# # PAGE: PLAYER PROFILE
# # ---------------------------------------------------------------------------
# def show_player_profile():
#     st.title("Player Profile")

#     # Team filter options
#     selected_team = st.sidebar.selectbox("Select Team", ["All Teams"] + all_teams)
    
#     # Filter players based on selected team
#     if selected_team != "All Teams":
#         team_bat_players = set(bat[bat["Team"] == selected_team]["Batter"])
#         team_bowl_players = set(bowl[bowl["Team"] == selected_team]["Bowler"])
#         team_players = sorted(list(team_bat_players.union(team_bowl_players)))
#     else:
#         team_players = all_players

#     if not team_players:
#         st.warning("No players found for the selected team.")
#         return

#     default_player = search if search in team_players else team_players[0]
    
#     # Reset index if the searched player is not in the filtered list
#     idx = team_players.index(default_player) if default_player in team_players else 0
#     player = st.sidebar.selectbox("Select Player", team_players, index=idx)

#     bcareer = bat_career[bat_career["Player"] == player]
#     wcareer = bowl_career[bowl_career["Player"] == player]

#     st.header(player)

#     c1, c2, c3, c4 = st.columns(4)
#     if not bcareer.empty:
#         row = bcareer.iloc[0]
#         c1.metric("Matches (Batting)", row["Matches"])
#         c2.metric("Runs", row["Runs"])
#         c3.metric("Highest Score", row["Highest Score"])
#         c4.metric("Batting Average", row["Average"])
#         c5, c6 = st.columns(2)
#         c5.metric("Strike Rate", row["Strike Rate"])
#         c6.metric("4s / 6s", f"{row['4s']} / {row['6s']}")
#     else:
#         st.info("No batting record for this player.")

#     if not wcareer.empty:
#         st.subheader("Bowling Summary")
#         row = wcareer.iloc[0]
#         b1, b2, b3, b4, b5 = st.columns(5)
#         b1.metric("Overs", row["Overs"])
#         b2.metric("Wickets", int(row["Wickets"]))
#         b3.metric("Economy", row["Economy"])
#         b4.metric("Bowling Average", row["Bowling Average"])
#         b5.metric("Dot %", row["Dot %"])

#     st.divider()

#     # --- Match History (Batting) ---
#     st.subheader("Match History (Batting)")
#     player_bat = bat[bat["Batter"] == player][
#         ["Match_ID", "Team", "Opposition", "Runs", "Balls", "4s", "6s", "SR", "Dismissal_Type"]
#     ].sort_values("Match_ID")
    
#     if not player_bat.empty:
#         st.dataframe(player_bat, width='stretch', hide_index=True)
#     else:
#         st.info("No batting match history available.")

#     st.divider()

#     # --- Match History (Bowling) ---
#     st.subheader("Match History (Bowling)")
#     player_bowl = bowl[bowl["Bowler"] == player][
#         ["Match_ID", "Team", "Opposition", "overs", "maidens", "runs", "wickets", "Economy", "0s"]
#     ].sort_values("Match_ID")
    
#     if not player_bowl.empty:
#         st.dataframe(player_bowl, width='stretch', hide_index=True)
#     else:
#         st.info("No bowling match history available.")

#     st.divider()

#     st.subheader("Performance Charts")
#     ch1, ch2 = st.columns(2)

#     with ch1:
#         if not player_bat.empty:
#             fig = px.line(player_bat, x="Match_ID", y="Runs", markers=True, title="Runs per Match")
#             st.plotly_chart(fig, width='stretch')
#         else:
#             st.info("No batting timeline data available.")

#     with ch2:
#         if not player_bat.empty:
#             fig = px.line(player_bat, x="Match_ID", y="SR", markers=True, title="Strike Rate Trend")
#             st.plotly_chart(fig, width='stretch')
#         else:
#             st.info("No strike rate timeline data available.")

#     ch3, ch4 = st.columns(2)
#     with ch3:
#         if not player_bowl.empty:
#             fig = px.bar(player_bowl, x="Match_ID", y="wickets", title="Wickets per Match")
#             st.plotly_chart(fig, width='stretch')
#         else:
#             st.info("No bowling charts available.")

#     with ch4:
#         if not player_bat.empty:
#             dismissal_counts = player_bat["Dismissal_Type"].value_counts().reset_index()
#             dismissal_counts.columns = ["Dismissal_Type", "Count"]
#             fig = px.pie(dismissal_counts, names="Dismissal_Type", values="Count", title="Dismissal Breakdown")
#             st.plotly_chart(fig, width='stretch')
#         else:
#             st.info("No dismissal data available.")

#     ch5, ch6 = st.columns(2)
#     with ch5:
#         if not player_bat.empty:
#             vs_opp = player_bat.groupby("Opposition")["Runs"].sum().reset_index().sort_values("Runs", ascending=False)
#             fig = px.bar(vs_opp, x="Opposition", y="Runs", title="Runs vs Opposition")
#             st.plotly_chart(fig, width='stretch')

#     with ch6:
#         if not player_bat.empty:
#             vs_ground = bat[bat["Batter"] == player].groupby("Ground")["Runs"].sum().reset_index().sort_values(
#                 "Runs", ascending=False)
#             if not vs_ground.empty:
#                 fig = px.bar(vs_ground, x="Ground", y="Runs", title="Runs vs Ground")
#                 st.plotly_chart(fig, width='stretch')


# # ---------------------------------------------------------------------------
# # PAGE: COMPARE PLAYERS
# # ---------------------------------------------------------------------------
# def show_compare():
#     st.title("Player Compare")

#     col1, col2 = st.columns(2)
#     player_a = col1.selectbox("Player A", all_players, index=0)
#     player_b = col2.selectbox("Player B", all_players, index=1 if len(all_players) > 1 else 0)

#     a_bat = bat_career[bat_career["Player"] == player_a]
#     b_bat = bat_career[bat_career["Player"] == player_b]
#     a_bowl = bowl_career[bowl_career["Player"] == player_a]
#     b_bowl = bowl_career[bowl_career["Player"] == player_b]

#     st.subheader("Batting Comparison")
#     metrics = ["Runs", "Average", "Strike Rate", "Highest Score"]
#     comp_rows = []
#     for m in metrics:
#         a_val = a_bat[m].iloc[0] if not a_bat.empty else 0
#         b_val = b_bat[m].iloc[0] if not b_bat.empty else 0
#         comp_rows.append({"Metric": m, player_a: a_val, player_b: b_val})
#     comp_df = pd.DataFrame(comp_rows)
#     st.dataframe(comp_df, width='stretch', hide_index=True)

#     fig = go.Figure()
#     fig.add_trace(go.Bar(name=player_a, x=comp_df["Metric"], y=comp_df[player_a]))
#     fig.add_trace(go.Bar(name=player_b, x=comp_df["Metric"], y=comp_df[player_b]))
#     fig.update_layout(barmode="group", title="Batting: Head to Head")
#     st.plotly_chart(fig, width='stretch')

#     st.subheader("Bowling Comparison")
#     b_metrics = ["Wickets", "Economy", "Bowling Average", "Dot %"]
#     comp_rows2 = []
#     for m in b_metrics:
#         a_val = a_bowl[m].iloc[0] if not a_bowl.empty else 0
#         b_val = b_bowl[m].iloc[0] if not b_bowl.empty else 0
#         comp_rows2.append({"Metric": m, player_a: a_val, player_b: b_val})
#     comp_df2 = pd.DataFrame(comp_rows2)
#     st.dataframe(comp_df2, width='stretch', hide_index=True)

#     fig2 = go.Figure()
#     fig2.add_trace(go.Bar(name=player_a, x=comp_df2["Metric"], y=comp_df2[player_a]))
#     fig2.add_trace(go.Bar(name=player_b, x=comp_df2["Metric"], y=comp_df2[player_b]))
#     fig2.update_layout(barmode="group", title="Bowling: Head to Head")
#     st.plotly_chart(fig2, width='stretch')


# # ---------------------------------------------------------------------------
# # PAGE: TEAM ANALYSIS
# # ---------------------------------------------------------------------------
# def show_team_analysis():
#     st.title("Team Analysis")
#     team = st.sidebar.selectbox("Select Team", all_teams)

#     st.header(team)
#     team_bat = bat_career[bat_career["Team"] == team].sort_values("Runs", ascending=False)
#     team_bowl = bowl_career[bowl_career["Team"] == team].sort_values("Wickets", ascending=False)

#     c1, c2 = st.columns(2)
#     with c1:
#         st.subheader("Batting Contribution")
#         fig = px.pie(team_bat, names="Player", values="Runs", title="Runs Share by Player")
#         st.plotly_chart(fig, width='stretch')
#         st.dataframe(team_bat, width='stretch', hide_index=True)

#     with c2:
#         st.subheader("Bowling Contribution")
#         fig = px.pie(team_bowl, names="Player", values="Wickets", title="Wickets Share by Player")
#         st.plotly_chart(fig, width='stretch')
#         st.dataframe(team_bowl, width='stretch', hide_index=True)


# # ---------------------------------------------------------------------------
# # ROUTER
# # ---------------------------------------------------------------------------
# if page == "Home":
#     show_home()
# elif page == "Player Profile":
#     show_player_profile()
# elif page == "Compare Players":
#     show_compare()
# elif page == "Team Analysis":
#     show_team_analysis()

Overwriting app.py


In [7]:
# %%writefile app.py
# import streamlit as st
# import pandas as pd
# import plotly.express as px
# import plotly.graph_objects as go

# st.set_page_config(page_title="Cricket Analytics Dashboard", page_icon="🏏", layout="wide")


# # ---------------------------------------------------------------------------
# # DATA LOADING + CLEANUP
# # ---------------------------------------------------------------------------
# @st.cache_data
# def load_data():
#     bat = pd.read_csv("Batting_Final.csv")
#     bowl = pd.read_csv("Bowling_Final.csv")
#     mm = pd.read_csv("Match_Master.csv")

#     for col in ["Team", "Batter"]:
#         bat[col] = bat[col].astype(str).str.strip()
#     for col in ["Team", "Bowler"]:
#         bowl[col] = bowl[col].astype(str).str.strip()

#     bat = bat[~bat["Batter"].str.contains(r"^-+$", regex=True, na=False)]
#     bowl = bowl[~bowl["Bowler"].str.contains(r"^-+$", regex=True, na=False)]
#     bat = bat.drop_duplicates().reset_index(drop=True)
#     bowl = bowl.drop_duplicates().reset_index(drop=True)

#     mm_small = mm[["Match_ID", "Tournament", "Ground", "Date", "Result", "Team 1", "Team 2"]]
#     bat = bat.merge(mm_small, on="Match_ID", how="left")
#     bowl = bowl.merge(mm_small, on="Match_ID", how="left")

#     def opposition(row):
#         if row["Team"] == row["Team 1"]:
#             return row["Team 2"]
#         return row["Team 1"]

#     bat["Opposition"] = bat.apply(opposition, axis=1)
#     bowl["Opposition"] = bowl.apply(opposition, axis=1)

#     return bat, bowl, mm


# bat, bowl, mm = load_data()


# # ---------------------------------------------------------------------------
# # CAREER AGGREGATES
# # ---------------------------------------------------------------------------
# @st.cache_data
# def build_batting_career(bat):
#     rows = []
#     for player, grp in bat.groupby("Batter"):
#         dismissed = grp[~grp["Dismissal_Type"].isin(["Not Out", "Retired Hurt"])]
#         not_outs = len(grp) - len(dismissed)
#         runs = grp["Runs"].sum()
#         balls = grp["Balls"].sum()
#         innings = len(grp)
#         outs = len(dismissed)
#         avg = round(runs / outs, 2) if outs > 0 else runs
#         sr = round(runs / balls * 100, 2) if balls > 0 else 0
#         rows.append({
#             "Player": player,
#             "Team": grp["Team"].iloc[-1],
#             "Matches": grp["Match_ID"].nunique(),
#             "Innings": innings,
#             "Runs": runs,
#             "Balls": balls,
#             "Highest Score": grp["Runs"].max(),
#             "Not Outs": not_outs,
#             "Average": avg,
#             "Strike Rate": sr,
#             "4s": grp["4s"].sum(),
#             "6s": grp["6s"].sum(),
#         })
#     return pd.DataFrame(rows)


# @st.cache_data
# def build_bowling_career(bowl):
#     rows = []
#     for player, grp in bowl.groupby("Bowler"):
#         overs = grp["overs"].sum()
#         balls = grp["balls"].sum()
#         runs = grp["runs"].sum()
#         wkts = grp["wickets"].sum()
#         econ = round(runs / overs, 2) if overs > 0 else 0
#         avg = round(runs / wkts, 2) if wkts > 0 else 0
#         sr = round(balls / wkts, 2) if wkts > 0 else 0
#         dot_pct = round(grp["0s"].sum() / balls * 100, 2) if balls > 0 else 0
#         rows.append({
#             "Player": player,
#             "Team": grp["Team"].iloc[-1],
#             "Matches": grp["Match_ID"].nunique(),
#             "Overs": round(overs, 1),
#             "Runs Conceded": runs,
#             "Wickets": wkts,
#             "Economy": econ,
#             "Bowling Average": avg,
#             "Bowling SR": sr,
#             "Dot %": dot_pct,
#         })
#     return pd.DataFrame(rows)


# bat_career = build_batting_career(bat)
# bowl_career = build_bowling_career(bowl)

# all_players = sorted(set(bat["Batter"]).union(set(bowl["Bowler"])))
# all_teams = sorted(list(set(bat["Team"].dropna().unique()).union(set(bowl["Team"].dropna().unique()))))
# all_tournaments = sorted(mm["Tournament"].dropna().unique())


# # ---------------------------------------------------------------------------
# # SIDEBAR NAVIGATION
# # ---------------------------------------------------------------------------
# st.sidebar.title("🏏 Cricket Analytics")
# page = st.sidebar.radio("Go to", ["Home", "Player Profile", "Compare Players", "Team Analysis"])

# search = st.sidebar.text_input("🔍 Search Player")
# if search:
#     matches = [p for p in all_players if search.lower() in p.lower()]
#     if matches:
#         st.sidebar.write(matches[:10])


# # ---------------------------------------------------------------------------
# # PAGE: HOME
# # ---------------------------------------------------------------------------
# def show_home():
#     st.title("🏏 Cricket Analytics Dashboard")

#     c1, c2, c3, c4 = st.columns(4)
#     c1.metric("Total Players", len(all_players))
#     c2.metric("Total Matches", mm["Match_ID"].nunique())
#     c3.metric("Total Runs", int(bat["Runs"].sum()))
#     c4.metric("Total Wickets", int(bowl["wickets"].sum()))

#     st.divider()
#     col1, col2 = st.columns(2)

#     with col1:
#         st.subheader("Top Run Scorers")
#         top_runs = bat_career.sort_values("Runs", ascending=False).head(10)
#         fig = px.bar(top_runs, x="Runs", y="Player", orientation="h", color="Runs")
#         fig.update_layout(yaxis={"categoryorder": "total ascending"})
#         st.plotly_chart(fig, width='stretch')

#     with col2:
#         st.subheader("Top Wicket Takers")
#         top_wkts = bowl_career.sort_values("Wickets", ascending=False).head(10)
#         fig = px.bar(top_wkts, x="Wickets", y="Player", orientation="h", color="Wickets")
#         fig.update_layout(yaxis={"categoryorder": "total ascending"})
#         st.plotly_chart(fig, width='stretch')


# # ---------------------------------------------------------------------------
# # PAGE: PLAYER PROFILE
# # ---------------------------------------------------------------------------
# def show_player_profile():
#     st.title("Player Profile")

#     # Team filter options
#     selected_team = st.sidebar.selectbox("Select Team", ["All Teams"] + all_teams)
    
#     # Filter players based on selected team
#     if selected_team != "All Teams":
#         team_bat_players = set(bat[bat["Team"] == selected_team]["Batter"])
#         team_bowl_players = set(bowl[bowl["Team"] == selected_team]["Bowler"])
#         team_players = sorted(list(team_bat_players.union(team_bowl_players)))
#     else:
#         team_players = all_players

#     if not team_players:
#         st.warning("No players found for the selected team.")
#         return

#     default_player = search if search in team_players else team_players[0]
    
#     idx = team_players.index(default_player) if default_player in team_players else 0
#     player = st.sidebar.selectbox("Select Player", team_players, index=idx)

#     bcareer = bat_career[bat_career["Player"] == player]
#     wcareer = bowl_career[bowl_career["Player"] == player]

#     st.header(player)

#     c1, c2, c3, c4 = st.columns(4)
#     if not bcareer.empty:
#         row = bcareer.iloc[0]
#         c1.metric("Matches (Batting)", row["Matches"])
#         c2.metric("Runs", row["Runs"])
#         c3.metric("Highest Score", row["Highest Score"])
#         c4.metric("Batting Average", row["Average"])
#         c5, c6 = st.columns(2)
#         c5.metric("Strike Rate", row["Strike Rate"])
#         c6.metric("4s / 6s", f"{row['4s']} / {row['6s']}")
#     else:
#         st.info("No batting record for this player.")

#     if not wcareer.empty:
#         st.subheader("Bowling Summary")
#         row = wcareer.iloc[0]
#         b1, b2, b3, b4, b5 = st.columns(5)
#         b1.metric("Overs", row["Overs"])
#         b2.metric("Wickets", int(row["Wickets"]))
#         b3.metric("Economy", row["Economy"])
#         b4.metric("Bowling Average", row["Bowling Average"])
#         b5.metric("Dot %", row["Dot %"])

#     st.divider()

#     # --- Match History (Batting) ---
#     st.subheader("Match History (Batting)")
#     player_bat = bat[bat["Batter"] == player][
#         ["Match_ID", "Date", "Ground", "Team", "Opposition", "Runs", "Balls", "4s", "6s", "SR", "Dismissal_Type"]
#     ].sort_values("Match_ID")
    
#     if not player_bat.empty:
#         st.dataframe(player_bat, width='stretch', hide_index=True)
#     else:
#         st.info("No batting match history available.")

#     st.divider()

#     # --- Match History (Bowling) ---
#     st.subheader("Match History (Bowling)")
#     player_bowl = bowl[bowl["Bowler"] == player][
#         ["Match_ID", "Date", "Ground", "Team", "Opposition", "overs", "maidens", "runs", "wickets", "Economy", "0s"]
#     ].sort_values("Match_ID")
    
#     if not player_bowl.empty:
#         st.dataframe(player_bowl, width='stretch', hide_index=True)
#     else:
#         st.info("No bowling match history available.")

#     st.divider()

#     st.subheader("Performance Charts")
#     ch1, ch2 = st.columns(2)

#     with ch1:
#         if not player_bat.empty:
#             fig = px.line(player_bat, x="Match_ID", y="Runs", markers=True, title="Runs per Match")
#             st.plotly_chart(fig, width='stretch')
#         else:
#             st.info("No batting timeline data available.")

#     with ch2:
#         if not player_bat.empty:
#             fig = px.line(player_bat, x="Match_ID", y="SR", markers=True, title="Strike Rate Trend")
#             st.plotly_chart(fig, width='stretch')
#         else:
#             st.info("No strike rate timeline data available.")

#     ch3, ch4 = st.columns(2)
#     with ch3:
#         if not player_bowl.empty:
#             fig = px.bar(player_bowl, x="Match_ID", y="wickets", title="Wickets per Match")
#             st.plotly_chart(fig, width='stretch')
#         else:
#             st.info("No bowling charts available.")

#     with ch4:
#         if not player_bat.empty:
#             dismissal_counts = player_bat["Dismissal_Type"].value_counts().reset_index()
#             dismissal_counts.columns = ["Dismissal_Type", "Count"]
#             fig = px.pie(dismissal_counts, names="Dismissal_Type", values="Count", title="Dismissal Breakdown")
#             st.plotly_chart(fig, width='stretch')
#         else:
#             st.info("No dismissal data available.")

#     ch5, ch6 = st.columns(2)
#     with ch5:
#         if not player_bat.empty:
#             vs_opp = player_bat.groupby("Opposition")["Runs"].sum().reset_index().sort_values("Runs", ascending=False)
#             fig = px.bar(vs_opp, x="Opposition", y="Runs", title="Runs vs Opposition")
#             st.plotly_chart(fig, width='stretch')

#     with ch6:
#         if not player_bat.empty:
#             vs_ground = bat[bat["Batter"] == player].groupby("Ground")["Runs"].sum().reset_index().sort_values(
#                 "Runs", ascending=False)
#             if not vs_ground.empty:
#                 fig = px.bar(vs_ground, x="Ground", y="Runs", title="Runs vs Ground")
#                 st.plotly_chart(fig, width='stretch')


# # ---------------------------------------------------------------------------
# # PAGE: COMPARE PLAYERS
# # ---------------------------------------------------------------------------
# def show_compare():
#     st.title("Player Compare")

#     col1, col2 = st.columns(2)
#     player_a = col1.selectbox("Player A", all_players, index=0)
#     player_b = col2.selectbox("Player B", all_players, index=1 if len(all_players) > 1 else 0)

#     a_bat = bat_career[bat_career["Player"] == player_a]
#     b_bat = bat_career[bat_career["Player"] == player_b]
#     a_bowl = bowl_career[bowl_career["Player"] == player_a]
#     b_bowl = bowl_career[bowl_career["Player"] == player_b]

#     st.subheader("Batting Comparison")
#     metrics = ["Runs", "Average", "Strike Rate", "Highest Score"]
#     comp_rows = []
#     for m in metrics:
#         a_val = a_bat[m].iloc[0] if not a_bat.empty else 0
#         b_val = b_bat[m].iloc[0] if not b_bat.empty else 0
#         comp_rows.append({"Metric": m, player_a: a_val, player_b: b_val})
#     comp_df = pd.DataFrame(comp_rows)
#     st.dataframe(comp_df, width='stretch', hide_index=True)

#     fig = go.Figure()
#     fig.add_trace(go.Bar(name=player_a, x=comp_df["Metric"], y=comp_df[player_a]))
#     fig.add_trace(go.Bar(name=player_b, x=comp_df["Metric"], y=comp_df[player_b]))
#     fig.update_layout(barmode="group", title="Batting: Head to Head")
#     st.plotly_chart(fig, width='stretch')

#     st.subheader("Bowling Comparison")
#     b_metrics = ["Wickets", "Economy", "Bowling Average", "Dot %"]
#     comp_rows2 = []
#     for m in b_metrics:
#         a_val = a_bowl[m].iloc[0] if not a_bowl.empty else 0
#         b_val = b_bowl[m].iloc[0] if not b_bowl.empty else 0
#         comp_rows2.append({"Metric": m, player_a: a_val, player_b: b_val})
#     comp_df2 = pd.DataFrame(comp_rows2)
#     st.dataframe(comp_df2, width='stretch', hide_index=True)

#     fig2 = go.Figure()
#     fig2.add_trace(go.Bar(name=player_a, x=comp_df2["Metric"], y=comp_df2[player_a]))
#     fig2.add_trace(go.Bar(name=player_b, x=comp_df2["Metric"], y=comp_df2[player_b]))
#     fig2.update_layout(barmode="group", title="Bowling: Head to Head")
#     st.plotly_chart(fig2, width='stretch')


# # ---------------------------------------------------------------------------
# # PAGE: TEAM ANALYSIS
# # ---------------------------------------------------------------------------
# def show_team_analysis():
#     st.title("Team Analysis")
#     team = st.sidebar.selectbox("Select Team", all_teams)

#     st.header(team)
#     team_bat = bat_career[bat_career["Team"] == team].sort_values("Runs", ascending=False)
#     team_bowl = bowl_career[bowl_career["Team"] == team].sort_values("Wickets", ascending=False)

#     c1, c2 = st.columns(2)
#     with c1:
#         st.subheader("Batting Contribution")
#         fig = px.pie(team_bat, names="Player", values="Runs", title="Runs Share by Player")
#         st.plotly_chart(fig, width='stretch')
#         st.dataframe(team_bat, width='stretch', hide_index=True)

#     with c2:
#         st.subheader("Bowling Contribution")
#         fig = px.pie(team_bowl, names="Player", values="Wickets", title="Wickets Share by Player")
#         st.plotly_chart(fig, width='stretch')
#         st.dataframe(team_bowl, width='stretch', hide_index=True)


# # ---------------------------------------------------------------------------
# # ROUTER
# # ---------------------------------------------------------------------------
# if page == "Home":
#     show_home()
# elif page == "Player Profile":
#     show_player_profile()
# elif page == "Compare Players":
#     show_compare()
# elif page == "Team Analysis":
#     show_team_analysis()

Overwriting app.py


In [7]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(page_title="Cricket Analytics Dashboard", page_icon="🏏", layout="wide")


# ---------------------------------------------------------------------------
# DATA LOADING + CLEANUP
# ---------------------------------------------------------------------------
@st.cache_data
def load_data():
    bat = pd.read_csv("Batting_Final.csv")
    bowl = pd.read_csv("Bowling_Final.csv")
    mm = pd.read_csv("Match_Master.csv")

    for col in ["Team", "Batter"]:
        bat[col] = bat[col].astype(str).str.strip()
    for col in ["Team", "Bowler"]:
        bowl[col] = bowl[col].astype(str).str.strip()

    bat = bat[~bat["Batter"].str.contains(r"^-+$", regex=True, na=False)]
    bowl = bowl[~bowl["Bowler"].str.contains(r"^-+$", regex=True, na=False)]
    bat = bat.drop_duplicates().reset_index(drop=True)
    bowl = bowl.drop_duplicates().reset_index(drop=True)

    # Match Master se exact 'Toss' aur 'Result' columns fetch kiye
    mm_small = mm[["Match_ID", "Tournament", "Ground", "Date", "Toss", "Result", "Team 1", "Team 2"]]
    mm_small["Date"] = pd.to_datetime(mm_small["Date"])
    bat = bat.merge(mm_small, on="Match_ID", how="left")
    bowl = bowl.merge(mm_small, on="Match_ID", how="left")

    def opposition(row):
        if row["Team"] == row["Team 1"]:
            return row["Team 2"]
        return row["Team 1"]

    bat["Opposition"] = bat.apply(opposition, axis=1)
    bowl["Opposition"] = bowl.apply(opposition, axis=1)

    return bat, bowl, mm


bat, bowl, mm = load_data()


# ---------------------------------------------------------------------------
# CAREER AGGREGATES
# ---------------------------------------------------------------------------
@st.cache_data
def build_batting_career(bat):
    rows = []
    for player, grp in bat.groupby("Batter"):
        dismissed = grp[~grp["Dismissal_Type"].isin(["Not Out", "Retired Hurt"])]
        not_outs = len(grp) - len(dismissed)
        runs = grp["Runs"].sum()
        balls = grp["Balls"].sum()
        innings = len(grp)
        outs = len(dismissed)
        avg = round(runs / outs, 2) if outs > 0 else runs
        st_rate = round(runs / balls * 100, 2) if balls > 0 else 0
        
        # Exact 'Batting_Hand' aur 'Bowling_Style' columns use kiye
        b_hand = grp["Batting_Hand"].dropna().iloc[0] if "Batting_Hand" in grp.columns and not grp["Batting_Hand"].dropna().empty else "N/A"
        b_style = grp["Bowling_Style"].dropna().iloc[0] if "Bowling_Style" in grp.columns and not grp["Bowling_Style"].dropna().empty else "N/A"

        rows.append({
            "Player": player,
            "Team": grp["Team"].iloc[-1],
            "Matches": grp["Match_ID"].nunique(),
            "Innings": innings,
            "Runs": runs,
            "Balls": balls,
            "Highest Score": grp["Runs"].max(),
            "Not Outs": not_outs,
            "Average": avg,
            "Strike Rate": st_rate,
            "4s": grp["4s"].sum(),
            "6s": grp["6s"].sum(),
            "Batting_Hand": b_hand,
            "Bowling_Style": b_style
        })
    return pd.DataFrame(rows)


@st.cache_data
def build_bowling_career(bowl):
    rows = []
    for player, grp in bowl.groupby("Bowler"):
        overs = grp["overs"].sum()
        balls = grp["balls"].sum()
        runs = grp["runs"].sum()
        wkts = grp["wickets"].sum()
        econ = round(runs / overs, 2) if overs > 0 else 0
        avg = round(runs / wkts, 2) if wkts > 0 else 0
        st_rate = round(balls / wkts, 2) if wkts > 0 else 0
        dot_pct = round(grp["0s"].sum() / balls * 100, 2) if balls > 0 else 0
        rows.append({
            "Player": player,
            "Team": grp["Team"].iloc[-1],
            "Matches": grp["Match_ID"].nunique(),
            "Overs": round(overs, 1),
            "Runs Conceded": runs,
            "Wickets": wkts,
            "Economy": econ,
            "Bowling Average": avg,
            "Bowling SR": st_rate,
            "Dot %": dot_pct,
        })
    return pd.DataFrame(rows)


bat_career = build_batting_career(bat)
bowl_career = build_bowling_career(bowl)

all_players = sorted(set(bat["Batter"]).union(set(bowl["Bowler"])))
all_teams = sorted(list(set(bat["Team"].dropna().unique()).union(set(bowl["Team"].dropna().unique()))))
all_tournaments = sorted(mm["Tournament"].dropna().unique())


# ---------------------------------------------------------------------------
# SIDEBAR NAVIGATION
# ---------------------------------------------------------------------------
st.sidebar.title("🏏 Cricket Analytics")
page = st.sidebar.radio("Go to", ["Home", "Player Profile", "Compare Players", "Team Analysis"])

search = st.sidebar.text_input("🔍 Search Player")
if search:
    matches = [p for p in all_players if search.lower() in p.lower()]
    if matches:
        st.sidebar.write(matches[:10])


# ---------------------------------------------------------------------------
# PAGE: HOME
# ---------------------------------------------------------------------------
def show_home():
    st.title("🏏 Cricket Analytics Dashboard")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Total Players", len(all_players))
    c2.metric("Total Matches", mm["Match_ID"].nunique())
    c3.metric("Total Runs", int(bat["Runs"].sum()))
    c4.metric("Total Wickets", int(bowl["wickets"].sum()))

    st.divider()
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Top Run Scorers")
        top_runs = bat_career.sort_values("Runs", ascending=False).head(10)
        fig = px.bar(top_runs, x="Runs", y="Player", orientation="h", color="Runs")
        fig.update_layout(yaxis={"categoryorder": "total ascending"})
        st.plotly_chart(fig, width='stretch')

    with col2:
        st.subheader("Top Wicket Takers")
        top_wkts = bowl_career.sort_values("Wickets", ascending=False).head(10)
        fig = px.bar(top_wkts, x="Wickets", y="Player", orientation="h", color="Wickets")
        fig.update_layout(yaxis={"categoryorder": "total ascending"})
        st.plotly_chart(fig, width='stretch')


# ---------------------------------------------------------------------------
# PAGE: PLAYER PROFILE
# ---------------------------------------------------------------------------
def show_player_profile():
    st.title("Player Profile")

    selected_team = st.sidebar.selectbox("Select Team", ["All Teams"] + all_teams)
    
    if selected_team != "All Teams":
        team_bat_players = set(bat[bat["Team"] == selected_team]["Batter"])
        team_bowl_players = set(bowl[bowl["Team"] == selected_team]["Bowler"])
        team_players = sorted(list(team_bat_players.union(team_bowl_players)))
    else:
        team_players = all_players

    if not team_players:
        st.warning("No players found for the selected team.")
        return

    default_player = search if search in team_players else team_players[0]
    
    idx = team_players.index(default_player) if default_player in team_players else 0
    player = st.sidebar.selectbox("Select Player", team_players, index=idx)

    bcareer = bat_career[bat_career["Player"] == player]
    wcareer = bowl_career[bowl_career["Player"] == player]

    st.header(player)
    if not bcareer.empty:
        hand = bcareer.iloc[0]["Batting_Hand"]
        style = bcareer.iloc[0]["Bowling_Style"]
        st.markdown(f"**Style:** 🏏 {hand} | 🥎 {style}")

    c1, c2, c3, c4 = st.columns(4)
    if not bcareer.empty:
        row = bcareer.iloc[0]
        c1.metric("Matches (Batting)", row["Matches"])
        c2.metric("Runs", row["Runs"])
        c3.metric("Highest Score", row["Highest Score"])
        c4.metric("Batting Average", row["Average"])
        c5, c6 = st.columns(2)
        c5.metric("Strike Rate", row["Strike Rate"])
        c6.metric("4s / 6s", f"{row['4s']} / {row['6s']}")
    else:
        st.info("No batting record for this player.")

    if not wcareer.empty:
        st.subheader("Bowling Summary")
        row = wcareer.iloc[0]
        b1, b2, b3, b4, b5 = st.columns(5)
        b1.metric("Overs", row["Overs"])
        b2.metric("Wickets", int(row["Wickets"]))
        b3.metric("Economy", row["Economy"])
        b4.metric("Bowling Average", row["Bowling Average"])
        b5.metric("Dot %", row["Dot %"])

    st.divider()

    # --- Match History (Batting) ---
    st.subheader("Match History (Batting)")
    player_bat = bat[bat["Batter"] == player][
        ["Match_ID", "Date", "Team", "Opposition", "Runs", "Balls", "4s", "6s", "SR", "Dismissal_Type", "Toss", "Result"]
    ].sort_values("Date")
    
    if not player_bat.empty:
        st.dataframe(player_bat, width='stretch', hide_index=True)
    else:
        st.info("No batting match history available.")

    st.divider()

    # --- Match History (Bowling) ---
    st.subheader("Match History (Bowling)")
    player_bowl = bowl[bowl["Bowler"] == player][
        ["Match_ID", "Date","Team", "Opposition", "overs", "maidens", "runs", "wickets", "Economy", "0s", "Toss", "Result"]
    ].sort_values("Date")
    
    if not player_bowl.empty:
        st.dataframe(player_bowl, width='stretch', hide_index=True)
    else:
        st.info("No bowling match history available.")

    st.divider()

    st.subheader("Performance Charts")
    ch1, ch2 = st.columns(2)

    with ch1:
        if not player_bat.empty:
            fig = px.line(player_bat, x="Match_ID", y="Runs", markers=True, title="Runs per Match")
            st.plotly_chart(fig, width='stretch')
        else:
            st.info("No batting timeline data available.")

    with ch2:
        if not player_bat.empty:
            fig = px.line(player_bat, x="Match_ID", y="SR", markers=True, title="Strike Rate Trend")
            st.plotly_chart(fig, width='stretch')
        else:
            st.info("No strike rate timeline data available.")

    ch3, ch4 = st.columns(2)
    with ch3:
        if not player_bowl.empty:
            fig = px.bar(player_bowl, x="Match_ID", y="wickets", title="Wickets per Match")
            st.plotly_chart(fig, width='stretch')
        else:
            st.info("No bowling charts available.")

    with ch4:
        if not player_bat.empty:
            dismissal_counts = player_bat["Dismissal_Type"].value_counts().reset_index()
            dismissal_counts.columns = ["Dismissal_Type", "Count"]
            fig = px.pie(dismissal_counts, names="Dismissal_Type", values="Count", title="Dismissal Breakdown")
            st.plotly_chart(fig, width='stretch')
        else:
            st.info("No dismissal data available.")

    ch5, ch6 = st.columns(2)
    with ch5:
        if not player_bat.empty:
            vs_opp = player_bat.groupby("Opposition")["Runs"].sum().reset_index().sort_values("Runs", ascending=False)
            fig = px.bar(vs_opp, x="Opposition", y="Runs", title="Runs vs Opposition")
            st.plotly_chart(fig, width='stretch')

    with ch6:
        if not player_bat.empty:
            vs_ground = bat[bat["Batter"] == player].groupby("Ground")["Runs"].sum().reset_index().sort_values(
                "Runs", ascending=False)
            if not vs_ground.empty:
                fig = px.bar(vs_ground, x="Ground", y="Runs", title="Runs vs Ground")
                st.plotly_chart(fig, width='stretch')


# ---------------------------------------------------------------------------
# PAGE: COMPARE PLAYERS
# ---------------------------------------------------------------------------
def show_compare():
    st.title("Player Compare")

    col1, col2 = st.columns(2)
    player_a = col1.selectbox("Player A", all_players, index=0)
    player_b = col2.selectbox("Player B", all_players, index=1 if len(all_players) > 1 else 0)

    a_bat = bat_career[bat_career["Player"] == player_a]
    b_bat = bat_career[bat_career["Player"] == player_b]
    a_bowl = bowl_career[bowl_career["Player"] == player_a]
    b_bowl = bowl_career[bowl_career["Player"] == player_b]

    st.subheader("Batting Comparison")
    metrics = ["Runs", "Average", "Strike Rate", "Highest Score"]
    comp_rows = []
    for m in metrics:
        a_val = a_bat[m].iloc[0] if not a_bat.empty else 0
        b_val = b_bat[m].iloc[0] if not b_bat.empty else 0
        comp_rows.append({"Metric": m, player_a: a_val, player_b: b_val})
    comp_df = pd.DataFrame(comp_rows)
    st.dataframe(comp_df, width='stretch', hide_index=True)

    fig = go.Figure()
    fig.add_trace(go.Bar(name=player_a, x=comp_df["Metric"], y=comp_df[player_a]))
    fig.add_trace(go.Bar(name=player_b, x=comp_df["Metric"], y=comp_df[player_b]))
    fig.update_layout(barmode="group", title="Batting: Head to Head")
    st.plotly_chart(fig, width='stretch')

    st.subheader("Bowling Comparison")
    b_metrics = ["Wickets", "Economy", "Bowling Average", "Dot %"]
    comp_rows2 = []
    for m in b_metrics:
        a_val = a_bowl[m].iloc[0] if not a_bowl.empty else 0
        b_val = b_bowl[m].iloc[0] if not b_bowl.empty else 0
        comp_rows2.append({"Metric": m, player_a: a_val, player_b: b_val})
    comp_df2 = pd.DataFrame(comp_rows2)
    st.dataframe(comp_df2, width='stretch', hide_index=True)

    fig2 = go.Figure()
    fig2.add_trace(go.Bar(name=player_a, x=comp_df2["Metric"], y=comp_df2[player_a]))
    fig2.add_trace(go.Bar(name=player_b, x=comp_df2["Metric"], y=comp_df2[player_b]))
    fig2.update_layout(barmode="group", title="Bowling: Head to Head")
    st.plotly_chart(fig2, width='stretch')


# ---------------------------------------------------------------------------
# PAGE: TEAM ANALYSIS
# ---------------------------------------------------------------------------
def show_team_analysis():
    st.title("Team Analysis")
    team = st.sidebar.selectbox("Select Team", all_teams)

    st.header(team)
    team_bat = bat_career[bat_career["Team"] == team].sort_values("Runs", ascending=False)
    team_bowl = bowl_career[bowl_career["Team"] == team].sort_values("Wickets", ascending=False)

    c1, c2 = st.columns(2)
    with c1:
        st.subheader("Batting Contribution")
        fig = px.pie(team_bat, names="Player", values="Runs", title="Runs Share by Player")
        st.plotly_chart(fig, width='stretch')
        st.dataframe(team_bat, width='stretch', hide_index=True)

    with c2:
        st.subheader("Bowling Contribution")
        fig = px.pie(team_bowl, names="Player", values="Wickets", title="Wickets Share by Player")
        st.plotly_chart(fig, width='stretch')
        st.dataframe(team_bowl, width='stretch', hide_index=True)


# ---------------------------------------------------------------------------
# ROUTER
# ---------------------------------------------------------------------------
if page == "Home":
    show_home()
elif page == "Player Profile":
    show_player_profile()
elif page == "Compare Players":
    show_compare()
elif page == "Team Analysis":
    show_team_analysis()

Overwriting app.py


In [9]:
import subprocess
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.headless', 'true'])
print("http://localhost:8501")

http://localhost:8501


In [10]:
# !pip install streamlit

In [ ]:
!pkill -f streamlit

In [10]:
!pip freeze > requirements.txt

In [11]:
!git --version

'git' is not recognized as an internal or external command,
operable program or batch file.


In [12]:
! pip install git

ERROR: Could not find a version that satisfies the requirement git (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for git
